# Notebook 01: Data Ingestion & Vector Store

Parse BMS 117 lecture PDFs, practice exam, answer key, and learning objectives.
Chunk text with metadata, then seed a ChromaDB collection via `chroma-mcp`.

## Setup

In [1]:
%pip install -q pymupdf sentence-transformers chromadb chroma-mcp openai openai-agents python-dotenv mcp


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import warnings
from pathlib import Path

import fitz  # PyMuPDF
from dotenv import load_dotenv

load_dotenv()

# Suppress noisy library warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*resource_tracker.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

import transformers
transformers.logging.set_verbosity_error()

/workspaces/final-project-oscar-peng/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Section 1: Define Data Paths

The project has three types of source material:
1. **Lecture PDFs** — the core knowledge base
2. **Practice Exam + Answer Key** — for evaluation
3. **Learning Objectives** — for curriculum alignment

In [11]:
# --- CONFIGURE THESE PATHS ---
# Point to wherever your data lives. Adjust if your repo structure differs.
DATA_DIR = Path("data")
LECTURES_DIR = DATA_DIR / "lectures"
EXAMS_DIR = DATA_DIR / "exams"
OBJECTIVES_DIR = DATA_DIR / "objectives"

# Create directories
for d in [LECTURES_DIR, EXAMS_DIR, OBJECTIVES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Lecture PDFs (edit this list to match your files)
LECTURE_FILES = {
    "Microbiology_Basics": "BMS117_Microbiology Basics_2022_ST.pdf",
    "Intro_Immune_System": "Intro_to_Immune_system_2025_STFINAL.pdf",
    "Overview_Immune_Response": "003_BMS117_Overview_IR_2025.pdf",
    "T_Cells": "005_T_cells_2025.pdf",
    "Humoral_Immunity": "Humoral_Immunity_Antibody_Structure_and_Function.pdf",
    "Effector_Mechanisms_Mucosal": "Antibody_Effector_Mechanisms__Mucosal_Immunity.pdf",
    "Staph_and_Strep": "Staph_and_Strep_2025_ST.pdf",
    "Hypersensitivity": "ILM_Hypersensitivity_Basics_2023.pdf",
    "Tolerance_Autoimmunity": "BMS_117_tolerance_and_autoimmunity_2024.pdf",
    "Immunodeficiency": "BMS_117_immunodeficiency_2024.pdf",
    "Pathogenesis_Host_Response": "010_Pathogenesis_and_Host_Response_2025_ST.pdf",
    "Bacterial_Genetics": "Bacterial_Genetics_2025.pdf",
    "Weekly_Recap_1": "Weekly_Recap_1_2025.pdf",
    "Weekly_Recap_2": "Weekly_Recap_2_2025.pdf",
}

EXAM_FILE = EXAMS_DIR / "BMS_117_Practice_Exam_1_2025.pdf"
KEY_FILE = EXAMS_DIR / "BMS_117_Practice_Exam_1_KEY_2025.pdf"
OBJECTIVES_FILE = OBJECTIVES_DIR / "Objectives_for_Exam_1_2025.pdf"

# Verify files exist
missing = []
for name, fname in LECTURE_FILES.items():
    if not (LECTURES_DIR / fname).exists():
        missing.append(f"  Lecture: {fname}")
for f in [EXAM_FILE, KEY_FILE, OBJECTIVES_FILE]:
    if not f.exists():
        missing.append(f"  {f}")

if missing:
    print("⚠️  Missing files (copy them into the data/ subdirectories):")
    print("\n".join(missing))
else:
    print(f"✓ All {len(LECTURE_FILES)} lectures + exam files found")

✓ All 14 lectures + exam files found


## Section 2: Extract Text from Lecture PDFs

Use PyMuPDF (`fitz`) to extract text from each lecture slide. Since these are lecture slides
(not dense documents), each page has relatively little text. We group consecutive pages
into chunks to get semantically meaningful passages.

In [12]:
def extract_pdf_text(filepath: Path) -> list[dict]:
    """Extract text from each page of a PDF. Returns list of {page, text}."""
    doc = fitz.open(filepath)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:  # skip blank pages
            pages.append({"page": i + 1, "text": text})
    doc.close()
    return pages


def chunk_pages(pages: list[dict], source: str, pages_per_chunk: int = 2) -> list[dict]:
    """Group consecutive pages into chunks with metadata.
    
    Lecture slides are sparse (~300 chars/page), so we combine 2 pages per chunk
    to get more meaningful retrieval units. Each chunk records its source lecture
    and page range for citation."""
    chunks = []
    for i in range(0, len(pages), pages_per_chunk):
        group = pages[i:i + pages_per_chunk]
        combined_text = "\n\n".join(p["text"] for p in group)
        page_start = group[0]["page"]
        page_end = group[-1]["page"]
        
        chunks.append({
            "id": f"{source}_p{page_start}-{page_end}",
            "text": combined_text,
            "source": source,
            "page_start": page_start,
            "page_end": page_end,
            "type": "lecture",
        })
    return chunks


# Process all lectures
all_chunks = []
for lecture_name, filename in LECTURE_FILES.items():
    filepath = LECTURES_DIR / filename
    pages = extract_pdf_text(filepath)
    chunks = chunk_pages(pages, source=lecture_name)
    all_chunks.extend(chunks)
    print(f"  {lecture_name:35s}  {len(pages):2d} pages → {len(chunks):2d} chunks")

print(f"\n✓ Total lecture chunks: {len(all_chunks)}")

  Microbiology_Basics                  24 pages → 12 chunks
  Intro_Immune_System                  31 pages → 16 chunks
  Overview_Immune_Response             17 pages →  9 chunks
  T_Cells                              38 pages → 19 chunks
  Humoral_Immunity                     20 pages → 10 chunks
  Effector_Mechanisms_Mucosal          19 pages → 10 chunks
  Staph_and_Strep                      39 pages → 20 chunks
  Hypersensitivity                     15 pages →  8 chunks
  Tolerance_Autoimmunity               33 pages → 17 chunks
  Immunodeficiency                     40 pages → 20 chunks
  Pathogenesis_Host_Response           33 pages → 17 chunks
  Bacterial_Genetics                   27 pages → 14 chunks
  Weekly_Recap_1                        3 pages →  2 chunks
  Weekly_Recap_2                        3 pages →  2 chunks

✓ Total lecture chunks: 176


## Section 3: Parse Learning Objectives

The objectives PDF lists learning goals organized by lecture topic. We parse each
section as a chunk so the RAG agent can cite relevant objectives.

In [13]:
def parse_objectives(filepath: Path) -> list[dict]:
    """Parse learning objectives PDF into chunks by topic section."""
    doc = fitz.open(filepath)
    full_text = "\n".join(page.get_text() for page in doc)
    doc.close()
    
    # Remove headers/footers
    full_text = re.sub(r"Exam 1 Session Objectives\s*", "", full_text)
    full_text = re.sub(r"Winter 2025\s*", "", full_text)
    full_text = re.sub(r"Draft Jan \d+, \d+\s*\d*", "", full_text)
    
    # Split on section headers (lines that don't start with bullet points)
    # Section headers are like 'ILM: The Microbial World...', 'Intro to the immune System', etc.
    sections = re.split(r"\n(?=[A-Z][A-Za-z\s:,\-/()]+\n•)", full_text)
    
    objective_chunks = []
    for section in sections:
        section = section.strip()
        if not section or len(section) < 30:
            continue
        
        # Extract section title (first line)
        lines = section.split("\n")
        title = lines[0].strip()
        body = "\n".join(lines[1:]).strip()
        
        if body:
            obj_id = re.sub(r'[^a-zA-Z0-9]', '_', title)[:40]
            objective_chunks.append({
                "id": f"obj_{obj_id}",
                "text": f"Learning Objectives for {title}:\n{body}",
                "source": f"Objectives - {title}",
                "type": "objective",
            })
    
    return objective_chunks


objective_chunks = parse_objectives(OBJECTIVES_FILE)
all_chunks.extend(objective_chunks)

print(f"Parsed {len(objective_chunks)} objective sections:")
for chunk in objective_chunks:
    print(f"  {chunk['source']}")

Parsed 26 objective sections:
  Objectives - ILM: The Microbial World: Intro to Bacterial structures and Normal Flora
  Objectives - Compare and contrast prokaryotic and eukaryotic cells with regard to their key
  Objectives - Identify prokaryotic structures that promote interactions with a host and describe
  Objectives - Intro to the immune System
  Objectives - Describe the role of the complement system in controlling infection and how the
  Objectives - Staphylococcus and Streptococcus
  Objectives - Describe the post-infection immunopathologies associated with Streptococcus
  Objectives - Humoral Immunity and Antibody Structure
  Objectives - T cell Diversity, Antigen Recognition and Immune Response
  Objectives - Overview of the Immune Response
  Objectives - Effector Mechanisms and Mucosal Immunity
  Objectives - Compare and contrast active and passive immunity
  Objectives - Give examples of molecules, enzymes, and cells that are important for
  Objectives - Demo Room: Steriliz

## Section 4: Parse Practice Exam Answer Key

The answer key is structured gold: each question has the correct answer marked `CORRECT`,
a lecture source, and a mapped learning objective. We parse this into structured data for
both retrieval (as chunks) and evaluation (as a ground-truth dataset).

In [14]:
def parse_answer_key(filepath: Path) -> list[dict]:
    """Parse the practice exam answer key into structured question records.
    
    Each record includes: question number, question text, correct answer letter,
    correct answer text, lecture source, and learning objective."""
    doc = fitz.open(filepath)
    full_text = "\n".join(page.get_text() for page in doc)
    doc.close()
    
    # Clean headers/footers
    full_text = re.sub(r"Practice Exam 1\s*BMS 117\s*", "", full_text)
    full_text = re.sub(r"Page \d+ of \d+\s*", "", full_text)
    
    # Split on question numbers
    parts = re.split(r"(?=(?:^|\n)\d{1,2}\.)", full_text)
    parts = [p.strip() for p in parts if p.strip() and re.match(r"\d{1,2}\.", p.strip())]
    
    questions = []
    for block in parts:
        q_match = re.match(r"(\d{1,2})\.\s*", block)
        q_num = int(q_match.group(1))
        
        # Find all answer options and their positions
        options = list(re.finditer(r"([a-e])\)\s*", block))
        
        # Find CORRECT marker
        correct_pos = block.find("CORRECT")
        
        # The correct answer is the last option letter appearing before CORRECT
        correct_letter = "?"
        if correct_pos >= 0 and options:
            for opt in options:
                if opt.start() < correct_pos:
                    correct_letter = opt.group(1)
        
        # Extract question stem (text before first option)
        if options:
            stem = block[q_match.end():options[0].start()].strip()
        else:
            stem = block[q_match.end():].strip()
        
        # Extract the correct answer text
        correct_text = ""
        for i, opt in enumerate(options):
            if opt.group(1) == correct_letter:
                # Text runs from after this option marker to CORRECT or next option
                start = opt.end()
                end = correct_pos if correct_pos > start else (
                    options[i + 1].start() if i + 1 < len(options) else len(block)
                )
                correct_text = block[start:end].replace("CORRECT", "").strip()
                break
        
        # Extract objective (text after 'Objective:')
        obj_match = re.search(r"[Oo]bjective[:\s]*(.+?)$", block, re.DOTALL)
        objective = re.sub(r"\s+", " ", obj_match.group(1).strip()) if obj_match else ""
        
        questions.append({
            "num": q_num,
            "stem": re.sub(r"\s+", " ", stem),
            "correct_letter": correct_letter,
            "correct_text": re.sub(r"\s+", " ", correct_text),
            "objective": objective,
            "full_block": block,  # keep raw text for the vector store
        })
    
    return questions


exam_questions = parse_answer_key(KEY_FILE)

print(f"Parsed {len(exam_questions)} exam questions\n")
for q in exam_questions[:5]:
    print(f"Q{q['num']:2d}: ({q['correct_letter']}) {q['correct_text'][:60]}")
    print(f"     Objective: {q['objective'][:70]}\n")

Parsed 51 exam questions

Q 1: (d) A rinse containing antibody-like molecules that bind to surf
     Objective: Identify prokaryotic structures that promote interactions with a host 

Q 2: (d) Gram (+) cells retain the crystal violet-iodine complex in t
     Objective: Compare and contrast the cell wall structure of Gram (+), Gram (-), an

Q 3: (b) coagulase positive
     Objective: Describe the role of coagulase in differentiating between Staph. aureu

Q 4: (c) The uptake of free DNA in the environment by competent bacte
     Objective: Describe 3 mechanisms that lead to horizontal transfer of DNA between 

Q 5: (b) Recognizing peptide displayed on an APC receptor
     Objective: Describe the different classes of receptors and their roles in transmi



## Section 5: Build Exam Chunks for the Vector Store

We store each exam question + answer + objective as a chunk in the vector store.
This lets the tutor retrieve relevant exam explanations when students ask about specific topics.

In [15]:
exam_chunks = []
for q in exam_questions:
    chunk_text = (
        f"Practice Exam Question {q['num']}:\n"
        f"{q['stem']}\n\n"
        f"Correct Answer: ({q['correct_letter']}) {q['correct_text']}\n\n"
        f"Learning Objective: {q['objective']}"
    )
    exam_chunks.append({
        "id": f"exam_q{q['num']}",
        "text": chunk_text,
        "source": f"Practice Exam Q{q['num']}",
        "type": "exam",
    })

all_chunks.extend(exam_chunks)
print(f"Added {len(exam_chunks)} exam chunks")
print(f"\n✓ Total chunks for vector store: {len(all_chunks)}")

Added 51 exam chunks

✓ Total chunks for vector store: 253


## Section 6: Save Processed Data

Save all chunks as JSONL for reproducibility, and save the exam questions separately
for the evaluation notebook.

In [16]:
# Save all chunks
CHUNKS_FILE = DATA_DIR / "chunks.jsonl"
with open(CHUNKS_FILE, "w") as f:
    for chunk in all_chunks:
        # Don't save the full_block field from exam questions in the chunk file
        record = {k: v for k, v in chunk.items() if k != "full_block"}
        f.write(json.dumps(record) + "\n")
print(f"✓ Saved {len(all_chunks)} chunks to {CHUNKS_FILE}")

# Save exam questions separately for evaluation
EXAM_DATA_FILE = DATA_DIR / "exam_questions.json"
exam_export = [
    {k: v for k, v in q.items() if k != "full_block"}
    for q in exam_questions
]
with open(EXAM_DATA_FILE, "w") as f:
    json.dump(exam_export, f, indent=2)
print(f"✓ Saved {len(exam_export)} exam questions to {EXAM_DATA_FILE}")

✓ Saved 253 chunks to data/chunks.jsonl
✓ Saved 51 exam questions to data/exam_questions.json


## Section 7: Embed & Store in ChromaDB via chroma-mcp

Following the pattern from Lecture 08 Demo 2, we seed a ChromaDB collection
using the `chroma-mcp` server. This is the same vector store the RAG agent
will connect to in Notebook 02.

In [17]:
import sys
import tempfile
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# chroma-mcp persistent storage directory
# Use a fixed path so Notebook 02 can connect to the same data
CHROMA_DIR = str(Path("data/chroma_db").resolve())
Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)

chroma_cmd = str(Path(sys.executable).parent / "chroma-mcp")
COLLECTION_NAME = "bms117_course_materials"

server_params = StdioServerParameters(
    command=chroma_cmd,
    args=["--client-type", "persistent", "--data-dir", CHROMA_DIR],
)

# Seed the collection with all chunks
# chroma-mcp handles embedding internally using its default model
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        
        # Create (or recreate) the collection
        try:
            await session.call_tool("chroma_delete_collection", {
                "collection_name": COLLECTION_NAME,
            })
        except Exception:
            pass  # collection didn't exist yet
        
        await session.call_tool("chroma_create_collection", {
            "collection_name": COLLECTION_NAME,
        })
        
        # Add chunks in batches (chroma-mcp has a limit per call)
        BATCH_SIZE = 20
        for i in range(0, len(all_chunks), BATCH_SIZE):
            batch = all_chunks[i:i + BATCH_SIZE]
            await session.call_tool("chroma_add_documents", {
                "collection_name": COLLECTION_NAME,
                "documents": [c["text"] for c in batch],
                "ids": [c["id"] for c in batch],
                "metadatas": [
                    {
                        "source": c["source"],
                        "type": c["type"],
                        "page_start": c.get("page_start", 0),
                        "page_end": c.get("page_end", 0),
                    }
                    for c in batch
                ],
            })
            print(f"  Batch {i // BATCH_SIZE + 1}: added {len(batch)} chunks")

print(f"\n✓ Seeded {len(all_chunks)} chunks into ChromaDB at {CHROMA_DIR}")

  Batch 1: added 20 chunks
  Batch 2: added 20 chunks
  Batch 3: added 20 chunks
  Batch 4: added 20 chunks
  Batch 5: added 20 chunks
  Batch 6: added 20 chunks
  Batch 7: added 20 chunks
  Batch 8: added 20 chunks
  Batch 9: added 20 chunks
  Batch 10: added 20 chunks
  Batch 11: added 20 chunks
  Batch 12: added 20 chunks
  Batch 13: added 13 chunks

✓ Seeded 253 chunks into ChromaDB at /workspaces/final-project-oscar-peng/data/chroma_db


## Section 8: Verify Retrieval

Quick sanity check — query the collection with a sample question to make sure
retrieval is working before moving to Notebook 02.

In [18]:
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        
        result = await session.call_tool("chroma_query_documents", {
            "collection_name": COLLECTION_NAME,
            "query_texts": ["What is bacterial transformation?"],
            "n_results": 3,
            "include": ["documents", "metadatas", "distances"],
        })
        
        print("Query: What is bacterial transformation?\n")
        print("Top 3 retrieved chunks:")
        print(result.content[0].text[:2000])

Query: What is bacterial transformation?

Top 3 retrieved chunks:
{"ids": [["exam_q4", "obj_Compare_and_contrast_between_point_mutat", "exam_q38"]], "embeddings": null, "documents": [["Practice Exam Question 4:\nWith regard to bacterial genetics, \u201cTransformation\u201d describes a process that involves:\n\nCorrect Answer: (c) The uptake of free DNA in the environment by competent bacterial cells\n\nLearning Objective: Describe 3 mechanisms that lead to horizontal transfer of DNA between bacterial cells and how they each impact genetic variability of bacteria", "Learning Objectives for Compare and contrast between point mutations, conjugation, transformation, and:\ntransduction and how each of these impacts (positive or negative) microbial \ngenomes\n\u2022", "Practice Exam Question 38:\nYou isolate a pathogenic E. coli strain producing a toxin and expose it to UV light, causing cell lysis. The lysate (retaining nucleic acids) is incubated with non- pathogenic E. coli. After plating

## Section 9: Summary Statistics

Overview of the ingested corpus.

In [19]:
from collections import Counter

type_counts = Counter(c["type"] for c in all_chunks)
total_chars = sum(len(c["text"]) for c in all_chunks)

print("=== Corpus Summary ===")
print(f"Total chunks:     {len(all_chunks)}")
print(f"Total characters: {total_chars:,}")
print(f"\nBy type:")
for t, count in type_counts.most_common():
    chars = sum(len(c["text"]) for c in all_chunks if c["type"] == t)
    print(f"  {t:15s}  {count:3d} chunks  ({chars:,} chars)")
print(f"\nExam questions:   {len(exam_questions)}")
print(f"ChromaDB path:    {CHROMA_DIR}")
print(f"\n✓ Data ready for Notebook 02 (RAG Agent)")

=== Corpus Summary ===
Total chunks:     253
Total characters: 144,820

By type:
  lecture          176 chunks  (112,472 chars)
  exam              51 chunks  (22,345 chars)
  objective         26 chunks  (10,003 chars)

Exam questions:   51
ChromaDB path:    /workspaces/final-project-oscar-peng/data/chroma_db

✓ Data ready for Notebook 02 (RAG Agent)
